# Phase M — Mechanical Displacement vs. Dielectric Phase Shift Discrimination
**An Empirical Confrontation, Forward Modeling, and Decision-Framework Evaluation**

Repeat-pass Synthetic Aperture Radar Interferometry (InSAR) measures an interferometric phase $\Delta \phi$ that is a scalar mixture of physical surface movement and electromagnetic scattering center variations:
$$\Delta \phi = \Delta \phi_{\text{disp}} + \Delta \phi_{\text{diel}} + \Delta \phi_{\text{atm}} + \Delta \phi_{\text{topo}} + \phi_{\text{noise}}$$

Over peatlands, **"bog breathing"** (poroelastic swelling and shrinkage) causes physical elevation changes that occur **in phase and with the same sign** as dielectric phase shifts caused by moisture variation in the upper *Sphagnum* capitulum layer.

### Objectives of this Phase:
1. **Part I (Empirical Analysis & Physical Forward Modeling)**:
   - Evaluate the Birchak-Debye-De Zan forward electromagnetic model against living peat physics.
   - Confront the live 90-date cumulative InSAR series against the physical desiccation ceiling ($6.13\text{ mm}$).
   - Examine hydrological driver coupling and time lags (instantaneous dielectric vs delayed poroelastic).
   - Test multi-temporal network baseline subsets to diagnose Zheng et al. (2022) fading closure-phase bias.
2. **Part II (Illustrative Synthetic Decision Frameworks)**:
   - Formulate the multi-geometry (2-LOS ascending/descending) error-propagation matrix.
   - Simulate the multi-frequency (C-band vs L-band) dispersion ratio as an operational discriminator.

In [ ]:
# --- Standard Project Bootstrap -------------------------------------------
from insar_wetlands.bootstrap import start
ctx = start("phaseM_mechanical_vs_dielectric")
log = ctx.log

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# Plot styling consistent with manuscript
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["legend.fontsize"] = 9
plt.rcParams["figure.dpi"] = 150

outdir = ctx.paths.outputs
outdir.mkdir(parents=True, exist_ok=True)
log.info(f"Phase M outputs will be saved to: {outdir}")

## PART I: EMPIRICAL ANALYSIS & VERIFIED PHYSICAL FORWARD MODELING

### Section 1: Complex Birchak-Debye-De Zan Forward Model
To determine whether dielectric variations can account for the observed $3.29\text{ mm}$ seasonal signal without requiring mechanical peat movement, we implement:
1. **Debye Relaxation**: Complex permittivity of free water $\epsilon_w(T, f)$ at $5.405\text{ GHz}$ across temperatures ($2^\circ\text{C}$ to $25^\circ\text{C}$).
2. **Birchak Refractive Mixing**: Living moss matrix (porosity $\phi = 93\%$, dry solid fraction $v_s = 0.07$, $\epsilon_s \approx 2.2$).
3. **Complex Vertical Wavenumber**: $k_z = \sqrt{k_0^2 \epsilon_{\text{eff}} - k_0^2 \sin^2\theta}$ and skin depth $\delta_p = 1 / (2 |\text{Im}(k_z)|)$.
4. **De Zan Lossy Half-Space Coherence**: $g = \frac{2\sqrt{\beta_1 \beta_2}j}{k_{z2}^* - k_{z1}}$, predicting apparent line-of-sight displacement $d_{\text{LOS}} = \frac{\lambda}{4\pi} \arg(g)$.

In [ ]:
# --- Section 1: Complex Birchak Forward Model (referee.py) -----------------
from insar_wetlands.referee import birchak_peat_forward_model

# 1. Base point evaluation at reference peat conditions (mv=0.85, vs=0.07, T=15C)
base_res = birchak_peat_forward_model(mv=0.85, vs=0.07, eps_solid=2.2, T=15.0, theta_deg=32.26)
log.info("Base Birchak Model (T=15°C, mv=0.85):")
for k, v in base_res.items():
    log.info(f"  {k}: {v}")

# 2. Moisture Excursion Sweep across temperatures
c_speed = 2.99792458e8
f_hz = 5.405e9
lam = c_speed / f_hz
theta = np.deg2rad(32.26)
k0 = 2.0 * np.pi / lam
kx = k0 * np.sin(theta)

def eps_w(T_c, f_val):
    e_inf = 4.9
    e_0 = 88.045 - 0.4147 * T_c + 6.295e-4 * T_c**2 + 1.075e-5 * T_c**3
    twopitau = 1.1109e-10 - 3.824e-12 * T_c + 6.938e-14 * T_c**2 - 5.096e-16 * T_c**3
    return complex(e_inf + (e_0 - e_inf) / (1.0 + 1j * twopitau * f_val))

def eps_p(mv_val, vs_val=0.07, eps_s=2.2, T_c=15.0):
    va = 1.0 - vs_val - mv_val
    n = mv_val * np.sqrt(eps_w(T_c, f_hz)) + vs_val * np.sqrt(eps_s) + va * 1.0
    return complex(n**2)

def calc_kz(eps_val):
    k = np.sqrt((k0**2) * eps_val - kx**2)
    return -k if np.imag(k) > 0 else k

def calc_delta_p(mv_val, T_c=15.0):
    kz = calc_kz(eps_p(mv_val, T_c=T_c))
    return float(1.0 / (2.0 * (-np.imag(kz))) * 1000.0)

def calc_apparent_los(mv1, mv2, T_c=15.0):
    k1 = calc_kz(eps_p(mv1, T_c=T_c))
    k2 = calc_kz(eps_p(mv2, T_c=T_c))
    b1, b2 = -np.imag(k1), -np.imag(k2)
    g = 2.0 * np.sqrt(b1 * b2) * 1j / (np.conj(k2) - k1)
    return float(np.angle(g) * lam / (4.0 * np.pi) * 1000.0)

# Moisture excursion sweep
dmv_vals = np.linspace(0.0, 0.40, 80)
los_disp_15 = [calc_apparent_los(0.85, 0.85 - dm, T_c=15.0) for dm in dmv_vals]
los_disp_2 = [calc_apparent_los(0.85, 0.85 - dm, T_c=2.0) for dm in dmv_vals]
los_disp_25 = [calc_apparent_los(0.85, 0.85 - dm, T_c=25.0) for dm in dmv_vals]

ceiling_val = float(base_res["asymptotic_ceiling_mm"])
env_mv_min = round(abs(float(base_res["envelope_los_mm"][0])), 2)
env_mv_max = round(abs(float(base_res["envelope_los_mm"][1])), 2)

# Compute joint envelope across dmv in [0.15, 0.35] and T in [2.0, 25.0] °C
joint_grid = [calc_apparent_los(0.85, 0.85 - dm, T_c=T_val)
              for dm in np.linspace(0.15, 0.35, 15)
              for T_val in np.linspace(2.0, 25.0, 15)]
joint_env_min = round(float(np.min(np.abs(joint_grid))), 2)
joint_env_max = round(float(np.max(np.abs(joint_grid))), 2)

print(f"C-band penetration depth at saturation (mv=0.85, T=15°C): {calc_delta_p(0.85):.2f} mm")
print(f"Apparent LOS shift for Δmv=0.25 (0.85 -> 0.60): {calc_apparent_los(0.85, 0.60):.2f} mm")
print(f"Theoretical asymptotic desiccation ceiling (mv -> 0): {ceiling_val:.2f} mm")
print(f"Moisture envelope (T=15°C, dmv in [0.15, 0.35]): [{env_mv_min}, {env_mv_max}] mm")
print(f"Joint moisture-temperature envelope: [{joint_env_min}, {joint_env_max}] mm")

### Section 2: Observational Confrontation (InSAR Time Series vs Desiccation Ceiling)
We confront the forward model predictions against the empirical cumulative displacement time series (90 Sentinel-1 acquisitions over 2022–2024; `phaseG_aggregate_series.csv`).

An unconstrained linear harmonic sinusoid fits a semi-amplitude of $3.286\text{ mm}$ (peak-to-peak swing of $6.572\text{ mm}$), nominally exceeding the theoretical desiccation ceiling ($6.13\text{ mm}$). However, fitting a physically bounded saturating model ($S \tanh(\cdot)$) resolves this discrepancy, contracting the peak-to-peak swing to $5.64\text{--}5.83\text{ mm}$ while preserving $R^2 = 0.364$.

In [ ]:
# --- Section 2: InSAR Cumulative Series & Saturating Seasonal Fit ----------
from insar_wetlands.referee import saturating_seasonal_fit

# Load empirical cumulative series
series_path = ctx.paths.tables / "phaseG_aggregate_series.csv"
if not series_path.exists():
    series_path = Path("results/tables/phaseG_aggregate_series.csv")
if not series_path.exists():
    series_path = Path("../../results/tables/phaseG_aggregate_series.csv")

df_series = pd.read_csv(series_path)
df_series["date"] = pd.to_datetime(df_series["date"])

# Fit linear harmonic vs free saturating vs ceiling-constrained tanh
fit_results = saturating_seasonal_fit(df_series, ceiling_mm=6.13)
summary_table = fit_results["summary_table"]
print("Table T16 — Comparison of Seasonal InSAR Models:")
print(summary_table.to_string(index=False))

# Reconstruct trajectories
d = df_series["date"]
t = (d - d.iloc[0]).dt.days.values / 365.25
y_obs = df_series["disp_mm"].values

lin_res = fit_results["linear_harmonic"]
sat_res = fit_results["ceiling_constrained"]

t_fine = np.linspace(t.min(), t.max(), 500)
d_fine = [d.iloc[0] + pd.Timedelta(days=float(val * 365.25)) for val in t_fine]
doy_pts = np.array([(date - pd.Timestamp("2022-01-01")).days / 365.25 for date in d_fine])

y_lin_curve = lin_res["semi_amplitude_mm"] * np.cos(2 * np.pi * doy_pts - np.deg2rad(lin_res["phase_doy"] / 365.25 * 360))
y_sat_curve = (ceiling_val / 2.0) * np.tanh(sat_res["semi_amplitude_mm"] * np.cos(2 * np.pi * doy_pts - np.deg2rad(sat_res["phase_doy"] / 365.25 * 360)) / (ceiling_val / 2.0))

### Section 3: Hydrological Driver Coupling & Suspect Epoch Screening [Illustrative Proxy for S2 NDWI]
The time lag between InSAR phase and hydrological forcing provides a critical diagnostic:
- **Dielectric response**: Near-instantaneous response of the upper 3–4 mm skin layer to precipitation/evaporation (lag $\approx 0$ days).
- **Mechanical peat consolidation**: Deep poroelastic drainage and pore-water pressure adjustment requiring multi-week consolidation (lag $\ge 12\text{--}36$ days).

We evaluate empirical correlations against optical surface wetness (Sentinel-2 NDWI) and Antecedent Precipitation Index (ERA5 API, $k=0.9$).

> **Data provenance & methodology note**: While Table T09 contains the genuine empirical correlation coefficients ($r = 0.450$ at lag 0) computed on the full Sentinel-2 cloud-masked stack on Google Drive, the raw per-date Sentinel-2 NDWI time series is not mirrored locally in `docs/paper/figures/`. To demonstrate the `dielectric_suspect_epochs` screening pipeline, this section generates an **illustrative harmonic proxy series** calibrated to the empirical seasonal cycle ($r \approx 0.45, \text{lag } 0$) and screens for simultaneous rapid subsidence and moisture drops.

In [ ]:
# --- Section 3: Hydrological Driver Coupling & Suspect Epoch Screening [Proxy] ---
from insar_wetlands.hydro import dielectric_suspect_epochs

t09_path = ctx.paths.tables / "T09_forcings.csv"
if not t09_path.exists():
    t09_path = Path("results/tables/T09_forcings.csv")
if not t09_path.exists():
    t09_path = Path("../../results/tables/T09_forcings.csv")

df_t09 = pd.read_csv(t09_path)
print("Table T09 — Empirical Driver Coupling on Raw vs Deseasonalized Anomalies:")
print(df_t09.to_string(index=False))

# Run dielectric_suspect_epochs on empirical InSAR series
# NOTE: Illustrative NDWI proxy calibrated to empirical seasonal cycle (r ≈ 0.45, lag 0)
doy = df_series["date"].dt.dayofyear
ndwi_site = pd.Series(0.45 + 0.15 * np.cos(2 * np.pi * (doy - 105) / 365.25), index=df_series["date"])
suspect_df = dielectric_suspect_epochs(df_series.set_index("date")["disp_mm"], ndwi_site, subsidence_mm=-5.0, ndwi_drop=-0.08)
n_suspect = int(suspect_df["dielectric_suspect"].sum())
print(f"\n[Illustrative Proxy] Dielectric suspect epochs identified: {n_suspect} / {len(suspect_df)}")
if n_suspect > 0:
    print(suspect_df[suspect_df["dielectric_suspect"]])

### Section 4: Multi-Temporal Baseline Subsets & Fading Bias Diagnostic
As demonstrated by Zheng et al. (2022), short temporal baseline networks ($\le 24\text{ d}$) suffer from systematic accumulating closure-phase bias (fading signal bias), manufacturing false subsidence rates ($-13.47\text{ mm yr}^{-1}$ in Table T15). Expanding the network to $\ge 48\text{ d}$ and annual pairs resolves the fading bias (collapsing velocity to $-1.53\text{ mm yr}^{-1}$), while the seasonal harmonic amplitude remains invariant ($2.89\text{--}3.29\text{ mm}$).

In [ ]:
# --- Section 4: Multi-Baseline Subsets (Table T15 / Zheng Fading Diagnostic) -
t15_path = ctx.paths.tables / "T15_subset_stability.csv"
if not t15_path.exists():
    t15_path = Path("results/tables/T15_subset_stability.csv")
if not t15_path.exists():
    t15_path = Path("../../results/tables/T15_subset_stability.csv")

df_t15 = pd.read_csv(t15_path)
print("Table T15 — Multi-Temporal Network Baseline Subset Stability:")
print(df_t15.to_string(index=False))

---
## ⚠️ PART II: ILLUSTRATIVE SYNTHETIC DECISION FRAMEWORKS ⚠️

> ### **CRITICAL SCIENTIFIC NOTIFICATION: SIMULATION ONLY — NOT AN EMPIRICAL MEASUREMENT**
> The analyses in **Module 5** below are **forward sensitivity simulations and mathematical error-propagation models**.
> - **No real descending-track SAR data** is inverted here: open ticket `X-021` (descending track HyP3 job) has not yet been processed.
> - **No real NISAR L-band observations** exist over this site: the NISAR satellite has not yet acquired an archive for this window.
> - This module demonstrates the mathematical criteria required for future multi-sensor separation. It **does NOT modify `paper_numbers.py`**, does NOT cite empirical p-values, and does NOT close any empirical replication tickets.
---

### Section 5: Multi-Geometry & Multi-Frequency Simulation Framework (Synthetic)
We formulate the mathematical decision frameworks that can unconditionally separate mechanical from dielectric signals once additional sensor geometries are acquired:

1. **2-LOS Matrix Decomposition**:
   $$\begin{pmatrix} d_{\text{LOS}}^{\text{asc}} \\ d_{\text{LOS}}^{\text{desc}} \end{pmatrix} = \begin{pmatrix} \cos \theta_{\text{asc}} & -\sin \theta_{\text{asc}} \cos \alpha_{\text{asc}} \\ \cos \theta_{\text{desc}} & \sin \theta_{\text{desc}} \cos \alpha_{\text{desc}} \end{pmatrix} \begin{pmatrix} d_{\text{vert}} \\ d_{\text{east}} \end{pmatrix}$$
2. **Multi-Frequency Dispersion Ratio (C- vs L-Band)**:
   $$\Delta \Phi_{\text{disp-free}} = \Delta \phi_C - \frac{\lambda_L}{\lambda_C} \Delta \phi_L$$
   Under pure mechanical motion, $\Delta \Phi_{\text{disp-free}} \equiv 0$. Under dielectric penetration variations, $\Delta \Phi_{\text{disp-free}} \neq 0$ because penetration depths scale non-linearly.

In [ ]:
# --- Section 5: Synthetic Multi-Geometry & Multi-Frequency Framework -------
# 1. 2-LOS Matrix Conditioning for Sentinel-1 Orbital Geometry
theta_asc = np.deg2rad(32.26)
alpha_asc = np.deg2rad(349.5)  # Heading angle ~-10.5 deg

theta_desc = np.deg2rad(34.10)
alpha_desc = np.deg2rad(190.5) # Heading angle ~190.5 deg

A_geom = np.array([
    [np.cos(theta_asc), -np.sin(theta_asc) * np.cos(alpha_asc)],
    [np.cos(theta_desc),  np.sin(theta_desc) * np.cos(alpha_desc)]
])
cond_A = np.linalg.cond(A_geom)
print(f"2-LOS Transformation Matrix A:\n{A_geom}")
print(f"Condition Number of A: {cond_A:.2f}")

# Simulate error propagation: 1 mm noise in LOS translates to vertical and horizontal uncertainty
cov_los = np.diag([1.0**2, 1.0**2])
cov_uh = np.linalg.inv(A_geom) @ cov_los @ np.linalg.inv(A_geom).T
sigma_vert = np.sqrt(cov_uh[0, 0])
sigma_east = np.sqrt(cov_uh[1, 1])
print(f"Error Propagation (1 mm LOS noise): σ_vert = {sigma_vert:.2f} mm, σ_east = {sigma_east:.2f} mm")

# 2. Multi-Frequency Dispersion Framework (C-band vs L-band)
f_C = 5.405e9
f_L = 1.250e9
lam_C = c_speed / f_C
lam_L = c_speed / f_L
freq_ratio = lam_L / lam_C  # ≈ 4.324

# Synthetic test cases:
# Case A: Pure mechanical vertical displacement (dz = 5.0 mm)
dz_true = 5.0
phi_C_disp = -4.0 * np.pi / lam_C * (dz_true * np.cos(theta_asc))
phi_L_disp = -4.0 * np.pi / lam_L * (dz_true * np.cos(theta_asc))
disp_free_A = phi_C_disp - (lam_L / lam_C) * phi_L_disp

# Case B: Pure dielectric phase shift (Δmv = 0.25)
phi_C_diel = np.deg2rad(base_res["point_estimate_los_mm"] / (lam_C * 1000.0) * 4.0 * np.pi)
R_CL_diel = 2.10 # Physical dielectric dispersion ratio (L-band penetrates ~5x deeper)
phi_L_diel = phi_C_diel / R_CL_diel
disp_free_B = phi_C_diel - (lam_L / lam_C) * phi_L_diel

print(f"\nMulti-Frequency Dispersion Metric (ΔΦ_disp-free):")
print(f"  Case A (Pure Displacement): ΔΦ_disp-free = {disp_free_A:.4f} rad (strictly 0)")
print(f"  Case B (Dielectric Phase Shift): ΔΦ_disp-free = {disp_free_B:.4f} rad (non-zero indicator)")

### Section 6: Unified Publication Dashboard & Provenance Archival
We synthesize all empirical and synthetic diagnostic panels into a unified multi-panel figure and record the formal execution provenance using `ctx.archive()`.

In [ ]:
# --- Section 6: Unified Dashboard & Archive --------------------------------
# Dynamic metrics lookup from loaded datasets and models (zero typed literals)
v_short_emp = float(df_t15.loc[df_t15["subset"] == "<=24d", "velocity_mm_yr"].iloc[0])
v_full_emp = float(df_t15.loc[df_t15["subset"] == "All pairs", "velocity_mm_yr"].iloc[0])
amp_full_emp = float(df_t15.loc[df_t15["subset"] == "All pairs", "amplitude_mm"].iloc[0])

fig = plt.figure(figsize=(15, 10), constrained_layout=True)
gs = fig.add_gridspec(2, 3)

# Panel A: Forward Dielectric Model
ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(dmv_vals, np.abs(los_disp_15), "b-", lw=2, label="T = 15°C (Base)")
ax0.fill_between(dmv_vals, np.abs(los_disp_2), np.abs(los_disp_25), color="b", alpha=0.15, label="2°C – 25°C Envelope")
ax0.axhline(ceiling_val, color="r", ls="--", lw=1.5, label=f"Desiccation Ceiling ({ceiling_val:.2f} mm)")
ax0.axhspan(env_mv_min, env_mv_max, color="green", alpha=0.15, label=f"Moisture Envelope ({env_mv_min}–{env_mv_max} mm)")
ax0.axhline(amp_full_emp, color="k", ls=":", lw=2, label=f"Observed Semi-Amplitude ({amp_full_emp:.2f} mm)")
ax0.set_xlabel("Volumetric Moisture Excursion (Δm_v)")
ax0.set_ylabel("Apparent InSAR Displacement (mm LOS)")
ax0.set_title("A. Dielectric Forward Model (Birchak-De Zan)")
ax0.legend(loc="lower right", fontsize=8)
ax0.set_ylim(0, 7.5)

# Panel B: Cumulative Time Series vs Saturating Fit
ax1 = fig.add_subplot(gs[0, 1])
ax1.scatter(df_series["date"], df_series["disp_mm"], color="#333333", s=20, alpha=0.7, label="Observed Cumulative (90 dates)")
ax1.plot(d_fine, y_lin_curve, "r--", lw=1.5, label=f"Linear Harmonic (P2P: {lin_res['peak_to_peak_mm']} mm)")
ax1.plot(d_fine, y_sat_curve, "g-", lw=2, label=f"Saturating tanh (P2P: {sat_res['peak_to_peak_mm']} mm)")
ax1.axhline(ceiling_val / 2.0, color="k", ls=":", alpha=0.5)
ax1.axhline(-ceiling_val / 2.0, color="k", ls=":", alpha=0.5)
ax1.set_ylabel("Cumulative Displacement (mm)")
ax1.set_title("B. InSAR Series vs Bounded Saturation")
ax1.legend(loc="lower left", fontsize=8)

# Panel C: Multi-Baseline Subset Fading Bias (Table T15)
ax2 = fig.add_subplot(gs[0, 2])
b_subsets = df_t15["subset"].tolist()
v_vals = df_t15["velocity_mm_yr"].values
amp_vals = df_t15["amplitude_mm"].values
ax2_twin = ax2.twinx()
p1 = ax2.plot(b_subsets, v_vals, "mo-", lw=2, label="Velocity (A−C)")
p2 = ax2_twin.plot(b_subsets, amp_vals, "cs-", lw=2, label="Seasonal Amplitude")
ax2.axhline(v_vals[-1], color="m", ls=":", alpha=0.5)
ax2.set_ylabel("Linear Velocity (mm/yr)", color="m")
ax2_twin.set_ylabel("Seasonal Semi-Amplitude (mm)", color="c")
ax2.set_title("C. Fading Signal Bias vs Harmonic Stability")
lines = p1 + p2
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, loc="center right", fontsize=8)

# Panel D: Multi-Geometry 2-LOS Error Sensitivity (Synthetic)
ax3 = fig.add_subplot(gs[1, 0])
angles = np.linspace(25, 45, 50)
conds = [np.linalg.cond([[np.cos(np.deg2rad(a)), -np.sin(np.deg2rad(a))*np.cos(alpha_asc)],
                        [np.cos(np.deg2rad(34.1)), np.sin(np.deg2rad(34.1))*np.cos(alpha_desc)]]) for a in angles]
ax3.plot(angles, conds, "k-", lw=2)
ax3.axvline(32.26, color="r", ls="--", label=f"S1 Track 22/168 (cond = {cond_A:.2f})")
ax3.set_xlabel("Ascending Incidence Angle θ_asc (°)")
ax3.set_ylabel("Matrix Condition Number cond(A)")
ax3.set_title("D. [Synthetic] 2-LOS Decomposition Geometry")
ax3.legend(loc="upper right", fontsize=8)

# Panel E: Dual-Frequency Dispersion Indicator (Synthetic)
ax4 = fig.add_subplot(gs[1, 1:])
scenarios = ["Pure Displacement\n(dz = 5 mm)", "Pure Dielectric Shift\n(Δmv = 0.25)", "Coupled Bog Breathing\n(Motion + Moisture)"]
disp_free_vals = [disp_free_A, disp_free_B, disp_free_A + disp_free_B]
bars = ax4.bar(scenarios, disp_free_vals, color=["#4daf4a", "#e41a1c", "#377eb8"], width=0.45, edgecolor="k", lw=1.2)
ax4.axhline(0, color="k", lw=1)
ax4.set_ylabel("Dispersion Residual ΔΦ_disp-free (rad)")
ax4.set_title("E. [Synthetic] Multi-Frequency (C vs L) Dispersion Test")
for bar in bars:
    yval = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2.0, yval + 0.05 * np.sign(yval), f"{yval:.3f} rad", ha="center", va="bottom" if yval>=0 else "top", fontweight="bold")
ax4.set_ylim(-2.0, 3.5)

# Save figure
fig_path = outdir / "mechanical_vs_dielectric_dashboard.png"
plt.savefig(fig_path, dpi=200)
print(f"Dashboard figure saved to: {fig_path}")

# Export summary metrics
summary_export = pd.DataFrame([
    {"metric": "observed_semi_amplitude_mm", "value": round(amp_full_emp, 3)},
    {"metric": "dielectric_envelope_min_mm", "value": env_mv_min},
    {"metric": "dielectric_envelope_max_mm", "value": env_mv_max},
    {"metric": "dielectric_joint_envelope_min_mm", "value": joint_env_min},
    {"metric": "dielectric_joint_envelope_max_mm", "value": joint_env_max},
    {"metric": "asymptotic_desiccation_ceiling_mm", "value": ceiling_val},
    {"metric": "saturating_model_p2p_mm", "value": round(float(sat_res["peak_to_peak_mm"]), 3)},
    {"metric": "saturating_model_r2", "value": round(float(sat_res["r2"]), 4)},
    {"metric": "zheng_fading_bias_short_baseline_mm_yr", "value": round(v_short_emp, 3)},
    {"metric": "full_network_velocity_mm_yr", "value": round(v_full_emp, 3)},
    {"metric": "synthetic_2los_cond_number", "value": round(float(cond_A), 2)},
])
csv_summary_path = outdir / "mechanical_vs_dielectric_summary.csv"
summary_export.to_csv(csv_summary_path, index=False)
print(f"Summary metrics exported to: {csv_summary_path}")

# Archive run
ctx.archive(
    params={"ceiling_mm": 6.13, "mv_nominal": 0.85, "theta_asc_deg": 32.26},
    products={"summary_csv": str(csv_summary_path.name), "dashboard_png": str(fig_path.name)}
)
print("Phase M successfully archived!")